In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    print("TPU available!")
    BATCH_SIZE = 128
except Exception:  # FIX: catch all TPU init errors (e.g. UnavailableError)
    print("No TPU. Using GPU.")
    strategy = tf.distribute.MirroredStrategy() if len(tf.config.list_physical_devices("GPU")) > 1 else tf.distribute.get_strategy()
    BATCH_SIZE = 32
    print(f"GPU count: {len(tf.config.list_physical_devices('GPU'))}")


In [ ]:
import os, glob

print("Searching for dataset...")
DATA_DIR = "/kaggle/input/datasets/belalsafy/egyptian-new-currency-2023/dataset"
if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train" in dirs:
            DATA_DIR = root
            break
    else:
        raise FileNotFoundError("Add egyptian-new-currency-2023 via Kaggle Add Input.")

print(f"DATA_DIR = {DATA_DIR}")
print(f"  train: {os.path.isdir(os.path.join(DATA_DIR, 'train'))}")
print(f"  valid: {os.path.isdir(os.path.join(DATA_DIR, 'valid'))}")
print(f"  test:  {os.path.isdir(os.path.join(DATA_DIR, 'test'))}")


In [ ]:
import os
import tensorflow as tf

try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/egyptian-new-currency-2023/dataset"
    if not os.path.exists(DATA_DIR):
        DATA_DIR = "./egyptian-new-currency-2023/dataset"
    print(f"DATA_DIR not set. Using fallback: {DATA_DIR}")

valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.gif')
data_dir = os.path.join(DATA_DIR, 'train')

print("Verifying image integrity...")
removed_count = 0

for root, dirs, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        if not file.lower().endswith(valid_extensions):
            try:
                os.remove(file_path)
                removed_count += 1
                continue
            except: pass
        try:
            img_bytes = tf.io.read_file(file_path)
            tf.io.decode_image(img_bytes)
        except Exception:
            print(f"Removing corrupted file: {file_path}")
            try:
                os.remove(file_path)
                removed_count += 1
            except: pass

print(f"Cleanup complete. Removed {removed_count} problematic files.")

## RUN 4: REVERT TO RUN 2 CONFIG + MORE AUGMENTATION
### Fix from Run 3 Catastrophe:
- REVERT to Run 2 architecture (no L2!)
- Keep LR at 1e-4 (NOT 5e-5)
- Add more augmentation only
- Same dropout rates as Run 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import datetime

IMG_HEIGHT = 128
IMG_WIDTH = 128

train_dir = os.path.join(DATA_DIR, "train")

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training",
    seed=123, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation",
    seed=123, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"Classes: {class_names}")
print(f"Number of classes: {NUM_CLASSES}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# Load held-out valid split for final test evaluation
valid_dir = os.path.join(DATA_DIR, "valid")
if os.path.isdir(valid_dir):
    test_ds = tf.keras.utils.image_dataset_from_directory(
        valid_dir, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)
    test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)
    print(f"Loaded held-out valid split: {len(test_ds)} batches")
else:
    test_ds = val_ds
    print("No valid/ split. Using val_ds for final evaluation.")

# One-hot encode labels for CategoricalCrossentropy
def to_one_hot(x, y):
    return x, tf.one_hot(tf.cast(y, tf.int32), NUM_CLASSES)

train_ds = train_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.map(to_one_hot,   num_parallel_calls=AUTOTUNE)
if test_ds is not val_ds:
    test_ds = test_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
print("One-hot encoding applied.")


In [ ]:
print("Run 4 already completed previously. Skipping to Run 5.")


In [ ]:
all_run_metrics = []
best_overall_acc = 0.6901
best_overall_run = 5
best_history = None
print("Run 4 already completed previously (best: 69.01%). Proceeding to Run 5.")


In [ ]:
print("Run 4 visualizations were saved in previous session. Skipping.")


In [ ]:
print("Run 4 accuracy/loss curves were saved in previous session. Skipping.")


In [ ]:
print("Run 4 classification report was saved in previous session. Skipping.")


In [ ]:
print("Run 4 model already saved from previous session. Skipping.")


## RUN 5: VGG-STYLE DEEP CNN TARGETING 93%
### Config:
- VGG-style: 4 blocks — 64→128→256→512 (2-3 Conv2D each)
- GlobalAveragePooling2D instead of Flatten (reduces overfitting)
- Input: 128×128 (faster training)
- 5 restarts × 50 epochs = max 250 epochs
- Adam(lr=1e-3) + CosineDecay → 1e-6
- CategoricalCrossentropy(label_smoothing=0.1)
- EarlyStopping(patience=10)
- Ensemble top 3 models


In [ ]:
import datetime
RUN5_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN5_DIR = f"./visualizations/run5_{RUN5_TIMESTAMP}"
os.makedirs(RUN5_DIR, exist_ok=True)
print(f"Run 5 outputs -> {RUN5_DIR}")
print(f"Classes: {class_names}")
print(f"Train: {len(train_ds)} batches, Val: {len(val_ds)} batches")

In [ ]:
def create_model_v5(num_classes):
    data_aug = models.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.1),
        layers.RandomTranslation(0.1, 0.1),
    ], name="augmentation")

    model = models.Sequential([
        layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        data_aug,
        layers.Rescaling(1./255),

        # Block 1: 64 x 2
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        # Block 2: 128 x 2
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        # Block 3: 256 x 3
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.3),

        # Block 4: 512 x 3
        layers.Conv2D(512, 3, padding="same", activation="relu"),
        layers.Conv2D(512, 3, padding="same", activation="relu"),
        layers.Conv2D(512, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.4),

        # Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])
    return model

EPOCHS_5 = 50
initial_lr = 1e-3
decay_steps = len(train_ds) * EPOCHS_5
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_lr, decay_steps=decay_steps, alpha=1e-6/initial_lr
)

# Build and display summary
model_v5 = create_model_v5(NUM_CLASSES)
model_v5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)
print("\n--- Run 5: VGG-Style Architecture (Target: 93%) ---")
model_v5.summary()


In [ ]:
NUM_RUNS_5 = 5

best5_acc = 0
best5_run = 0
best5_model = None
best5_history = None
all5_metrics = []
top3_models = []

print(f"{"="*60}")
print(f"RUN 5: VGG-STYLE | {NUM_RUNS_5} RUNS x {EPOCHS_5} EPOCHS")
print(f"Optimizer: Adam + CosineDecay (1e-3 -> 1e-6)")
print(f"Loss: CategoricalCrossentropy(label_smoothing=0.1)")
print(f"Target: 93%")
print(f"{"="*60}\n")

for run_num in range(1, NUM_RUNS_5 + 1):
    print(f"{"="*60}")
    print(f"RUN {run_num}/{NUM_RUNS_5}")
    print(f"{"="*60}")

    m = create_model_v5(NUM_CLASSES)
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"],
    )

    es = callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1)
    cp = callbacks.ModelCheckpoint(f"{RUN5_DIR}/run{run_num}_best.keras", monitor="val_accuracy", save_best_only=True, verbose=0)

    history = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_5, callbacks=[es, cp], verbose=1)

    val_acc = history.history["val_accuracy"]
    train_acc = history.history["accuracy"]
    best_val = max(val_acc)
    best_ep = val_acc.index(best_val) + 1
    final_val = val_acc[-1]
    final_train = train_acc[-1]
    epochs_run = len(val_acc)

    metrics = {"run": run_num, "best": best_val, "epoch": best_ep, "final_val": final_val, "final_train": final_train, "epochs": epochs_run}
    all5_metrics.append(metrics)

    marker = ""
    if best_val > best5_acc:
        best5_acc = best_val
        best5_run = run_num
        best5_model = m
        best5_history = history
        marker = " <<< BEST"
    if best_val >= 0.93:
        marker += " *** TARGET MET! ***"

    print(f"  Best={best_val:.4f} (ep{best_ep})  Final={final_val:.4f}  Train={final_train:.4f}{marker}")

    top3_models.append((best_val, m))
    top3_models.sort(key=lambda x: -x[0])
    top3_models = top3_models[:3]

    if run_num % 2 == 0 or run_num == NUM_RUNS_5:
        print(f"\n>>> PROGRESS <<<")
        for m2 in all5_metrics:
            print(f"  Run {m2["run"]}: Best={m2["best"]:.4f} (ep{m2["epoch"]})")
        print(f"  >> Current best: {best5_acc:.4f} | Target: 0.93 | Gap: {0.93 - best5_acc:.4f}")
        print(f"  >> Ensemble (top 3): {', '.join(f'{x[0]:.4f}' for x in top3_models)}\n")

print(f"\n{"="*60}")
print(f"RUN 5: COMPLETE")
print(f"{"="*60}")
print(f"Best single model: {best5_acc:.4f} (Run {best5_run})")
print(f"Top 3 ensemble: {', '.join(f'{x[0]:.4f}' for x in top3_models)}")
print(f"Target: 0.93 | Gap: {0.93 - best5_acc:.4f}")


In [ ]:
import numpy as np

run_nums5 = [m.get('run') for m in all5_metrics]
best5_vals = [m.get('best') for m in all5_metrics]
final5_vals = [m.get('final_val') for m in all5_metrics]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(run_nums5, best5_vals, 'b-o', label='Best Val Acc', linewidth=2, markersize=8)
axes[0].plot(run_nums5, final5_vals, 'r--s', label='Final Val Acc', linewidth=2, markersize=8)
axes[0].axhline(y=0.93, color='g', linestyle=':', linewidth=2, label='Target: 93%')
axes[0].set_xlabel('Run Number', fontsize=12)
axes[0].set_ylabel('Validation Accuracy', fontsize=12)
axes[0].set_title(f'Run 5: All {NUM_RUNS_5} Runs', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(1, NUM_RUNS_5 + 1))

axes[1].bar(run_nums5, best5_vals, color='steelblue', alpha=0.7)
axes[1].axhline(y=0.93, color='r', linestyle='--', linewidth=2, label='Target: 93%')
axes[1].axhline(y=best5_acc, color='darkorange', linestyle='-', linewidth=2, label=f'Best: {best5_acc:.4f}')
axes[1].set_xlabel('Run Number', fontsize=12)
axes[1].set_ylabel('Best Validation Accuracy', fontsize=12)
axes[1].set_title('Best Accuracy per Run', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_xticks(range(1, NUM_RUNS_5 + 1))

plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best: {best5_acc:.4f} (Run {best5_run}) | Target: 0.93 | Gap: {0.93 - best5_acc:.4f}")

In [ ]:
acc5 = best5_history.history["accuracy"]
val_acc5 = best5_history.history["val_accuracy"]
loss5 = best5_history.history["loss"]
val_loss5 = best5_history.history["val_loss"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc5, label="Training Accuracy", linewidth=2)
axes[0].plot(val_acc5, label="Validation Accuracy", linewidth=2)
axes[0].axhline(y=0.93, color="g", linestyle="--", label="Target: 93%")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Accuracy", fontsize=12)
axes[0].set_title(f"Run 5 Accuracy Curves (Best Run: {best5_run})", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss5, label="Training Loss", linewidth=2)
axes[1].plot(val_loss5, label="Validation Loss", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Loss", fontsize=12)
axes[1].set_title("Run 5 Loss Curves", fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_accuracy_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n--- RUN 5 Classification Report (Best Model on Test Set) ---")
y_true = np.argmax(np.concatenate([y for x, y in test_ds], axis=0), axis=1)
y_pred_best = np.argmax(best5_model.predict(test_ds), axis=-1)
test_acc = np.mean(y_pred_best == y_true)
print(f"Test Accuracy (best single model): {test_acc:.4f}")
print(classification_report(y_true, y_pred_best, target_names=class_names))

cm = confusion_matrix(y_true, y_pred_best)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.title("Run 5 Confusion Matrix (Best Single Model)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
print("\n--- RUN 5 Ensemble Prediction (Top 3 Models on Test Set) ---")
all_preds = []
for val, m in top3_models:
    p = m.predict(test_ds, verbose=0)
    all_preds.append(p)
    print(f"  Model (val_acc={val:.4f}): included")
avg_preds = np.mean(all_preds, axis=0)
y_pred_ensemble = np.argmax(avg_preds, axis=-1)
from sklearn.metrics import accuracy_score
ensemble_acc = accuracy_score(y_true, y_pred_ensemble)
print(f"\nEnsemble (top 3) Test Accuracy: {ensemble_acc:.4f}")
print(f"Best Single Model Test Accuracy: {test_acc:.4f}")
improvement = ensemble_acc - test_acc
print(f"Ensemble Improvement: +{improvement:.4f}")

if ensemble_acc >= 0.93:
    print(f"\n{"="*60}")
    print(f"*** TARGET 93% MET! Ensemble accuracy: {ensemble_acc:.4f} ***")
    print(f"{"="*60}")
else:
    print(f"\nTarget: 0.93 | Ensemble Gap: {0.93 - ensemble_acc:.4f}")

# Ensemble confusion matrix
cm_ens = confusion_matrix(y_true, y_pred_ensemble)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_ens, annot=True, fmt="d", cmap="Greens", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.title("Run 5 Confusion Matrix (Ensemble Top 3)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_ensemble_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
best5_model.save(f"{RUN5_DIR}/best_model_run5.keras")
for i, (val, m) in enumerate(top3_models):
    m.save(f"{RUN5_DIR}/ensemble_model_{i+1}_{val:.4f}.keras")

run_results_5 = []
for m2 in all5_metrics:
    run_results_5.append(f"Run {m2["run"]}: Best={m2["best"]:.4f} (ep{m2["epoch"]}), Final={m2["final_val"]:.4f}")

ensemble_str = f"Ensemble (top 3): {', '.join(f'{x[0]:.4f}' for x in top3_models)}"
result_text_5 = "\n".join(run_results_5 + [ensemble_str])

final_acc = max(test_acc, ensemble_acc)

run5_metrics = f"""### Run 5 (VGG-Style Architecture - Target 93%)
**Date**: {RUN5_TIMESTAMP}
**Status**: {'COMPLETED - TARGET MET! :rocket:' if final_acc >= 0.93 else 'COMPLETED - Below Target'}

### Configuration
- Architecture: VGG-style — 4 conv blocks (64→128→256→512), GlobalAveragePooling2D head
- Augmentation: Flip, Rotation(0.2), Zoom(0.15), Brightness(0.1), Contrast(0.1), Translation(0.1)
- Optimizer: Adam(lr=1e-3) + CosineDecay LR (1e-3 → 1e-6)
- Loss: CategoricalCrossentropy(label_smoothing=0.1)
- Dropout: 0.2, 0.25, 0.3, 0.4 (conv blocks), 0.5, 0.3 (dense)
- Epochs per run: {EPOCHS_5}
- Number of runs: {NUM_RUNS_5}
- Input size: 128x128

### Results (All {NUM_RUNS_5} Runs)
{result_text_5}

### Best Result
- **Best Single Model Test Accuracy**: {test_acc:.4f} (Run {best5_run})
- **Ensemble Test Accuracy (Top 3)**: {ensemble_acc:.4f}
- **Target**: 0.93 (93%)
- **Gap**: {0.93 - final_acc:.4f}
- Target Met: {'YES :rocket:' if final_acc >= 0.93 else 'NO'}

---
"""

with open("RUN_TRACKING.md", "a") as f:
    f.write(run5_metrics)

print(f"\nRun 5 metrics saved to RUN_TRACKING.md")
print(f"Visualizations saved to {RUN5_DIR}")
print(f"Best model saved to {RUN5_DIR}/best_model_run5.keras")
print(f"Ensemble models saved to {RUN5_DIR}/ensemble_model_*.keras")
